## 1. Импорты

In [ ]:
import os
import gc
import sys
import json
import toml
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import matplotlib.pyplot as plt
import tqdm
from prefect.blocks.system import Secret
from prefect.variables import Variable
import plotly.io as pio
from scipy.stats import linregress

import lightgbm as lgb
from sklearn.metrics import roc_auc_score

pio.renderers.default = "plotly_mimetype"

In [ ]:
!pip install hyperopt

In [ ]:
import lightgbm as lgb
from sklearn.feature_selection import RFE
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

## 2. Параметры

In [ ]:
secret_cdw = await Secret.load("pass-sb-analytics")
secret_space = await Secret.load("pass-space")
secret_sb_dirkaim = (await Secret.load("pass-sb-dirkaim")).get()

In [ ]:
SUBTASK = 'v2'             # Подзадача (или версия, свободное наименование)
COMMENT = ""               # Текст-подсказка в свободной форме, пояснение для разработчика
SEED = 41                  # random_state

USERNAME_SPACE = "analytics"
USERNAME_CDW = "sb_analytics"
USERNAME_SB_DIRKAIM = secret_sb_dirkaim["login"]
PASSWORD_SPACE = secret_space.get()
PASSWORD_CDW = secret_cdw.get()
PASSWORD_SB_DIRKAIM = secret_sb_dirkaim["password"]


common_params = toml.load('common_params.toml')

AUTHOR = common_params['AUTHOR']        # Разработчик
PREFIX = common_params['PREFIX']        # Префикс
TASK_NUM = common_params['TASK_NUM']    # Номер задачи eservices
OUT_TABLE = common_params['OUT_TABLE']  # out_table
ML_TYPE = 'reg'      # Тип ML-задачи: reg - регрессия, clf - классификация clf, mclf - мультирег,mclf - мультикласс
TARGET = common_params['TARGET']        # Имя целевой переменной
TIME_COL = common_params['TIME_COL']    # Имя колонки с датами

PRIMARY_KEY = ['client_did', 'report_date', 'contract_uid']       # Объект предсказания, ключ

# Технические поля - не фичи, не id, не дата, для удаления
TECH_COLS = [
    'ub_clients_registration_dt_diff_report_date',
    'ub_clients_client_email',
    'ub_clients_mailing_consent',
    "card_report_date_diff_card_work_dt_aggmin", # ЛИК в фиче! 18.12.2025
    'transaction_total_trans_cnt_1m',
    'transaction_ratio_cnt_trans_1m_to_1m',
    'transaction_supermarkets_trans_cnt_12m'
]


# Пути (для чтения файлов)
PATH_TO_UTILS = common_params['PATH_TO_UTILS'] # Путь к кастомной библиотеке toolbox

# Пути (для сохранения файлов)
DATA_FOLDER = common_params['DATA_FOLDER']            # Папка с выборками
MODELS_FOLDER = common_params['MODELS_FOLDER']        # Файлы модели
FEATURES_FOLDER = common_params['FEATURES_FOLDER']    # Папка с фичами
REPORTS_FOLDER = f'{MODELS_FOLDER}/{SUBTASK}/report'  # Папка с отчетами

# Пути к файлам
PATH_TO_SAMPLE = f'./{DATA_FOLDER}/sample.parquet'    # Исходная выборка для разработки
PATH_MODELS_DATA = f"./{MODELS_FOLDER}/{SUBTASK}"     # Папка с файлми модели

In [ ]:
print(PATH_TO_SAMPLE, FEATURES_FOLDER, MODELS_FOLDER)
print(PATH_MODELS_DATA)
print(REPORTS_FOLDER)

In [ ]:
# Создаем необходимые директории
os.makedirs(MODELS_FOLDER, exist_ok=True)
os.makedirs(PATH_MODELS_DATA, exist_ok=True)
os.makedirs(REPORTS_FOLDER, exist_ok=True)

### Загрузка кастомного модуля

In [ ]:
sys.path.append(PATH_TO_UTILS)

In [ ]:
!pip install openpyxl # для from toolbox.modeling_utils.report

In [ ]:
from toolbox.modeling_utils import preprocessing 
from toolbox.modeling_utils import report
from toolbox.modeling_utils import feature_selection
from toolbox.modeling_utils import report_generator
from toolbox import oracle

## 3. Логгирование

In [ ]:
# Логгирование
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt = "%Y-%m-%d %H:%M",
    handlers = [
        # logging.FileHandler("logfile.log"),
        logging.StreamHandler(),
    ])

In [ ]:
!pip install clearml

In [ ]:
os.environ["CLEARML_AUTO_LOG_ARTIFACTS"] = "0" 
os.environ["CLEARML_MONITOR_SYSTEM"]    = "0"
os.environ["CLEARML_AUTO_CONNECT_FRAMEWORKS"] = "0"

import clearml
from clearml import Task, Logger, Model

PROJECT_NAME = f"eservices-{TASK_NUM}"

clearml_task = Task.init(
    project_name=PROJECT_NAME,
    task_name=f"{PROJECT_NAME}_{SUBTASK}",
    auto_connect_streams=False,
    reuse_last_task_id=False
)

## 4 Чтение данных

In [ ]:
filenames = list(filter(
    lambda file: file.startswith(PREFIX.lower()) and file.endswith('.parquet'),
    os.listdir(FEATURES_FOLDER)
))
feature_paths = [f'{FEATURES_FOLDER}/{filename}' for filename in filenames]

sample = pd.read_parquet(PATH_TO_SAMPLE)

In [ ]:
sample = sample[sample['report_date']>='2025-09-01']

In [ ]:
sample

## 4. EDA

In [ ]:
# Динамика таргета во времени
sample['year_month'] = sample[TIME_COL].dt.strftime('%Y-%m-01')
sample['year_month'] = pd.to_datetime(sample['year_month'])

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
display(sample.pivot_table(values=['client_did'],
                index=['year_month'],
                aggfunc='count',
                margins=False).T)

pt = (sample
    .pivot_table(
        values=["client_did"],
        index=["year_month"],
        aggfunc="count",
        margins=False,).reset_index())

fig_sample = px.bar(
    pt,
    x="year_month",
    y="client_did",
    orientation="v",
    title="Количество клиентов по месяцам",
)

fig_sample.show()  # в ноутбуке — опционально

In [ ]:
Logger.current_logger().report_plotly(
    title="001 Кол-во клиентов в выборке по месяцам",
    series="",
    figure=fig_sample)

In [ ]:
display(sample.pivot_table(values=['contract_uid'],
                index=['year_month'],
                aggfunc='count',
                margins=False).T)

pt = (sample
    .pivot_table(
        values=["contract_uid"],
        index=["year_month"],
        aggfunc="count",
        margins=False,).reset_index())

fig_sample = px.bar(
    pt,
    x="year_month",
    y="contract_uid",
    orientation="v",
    title="Количество контрактов по месяцам",
)

fig_sample.show()  # в ноутбуке — опционально

In [ ]:
Logger.current_logger().report_plotly(
    title="001 Кол-во контрактов в выборке по месяцам",
    series="",
    figure=fig_sample)

In [ ]:
from typing import Tuple, Optional, Sequence, List, Any
def counts_target_ratio_sample(
    data: pd.DataFrame,
    date_col: str,
    target_col: str,
    title: str = "Размеры и mean Target по месяцам",
    count_series_name: str = "Target count",
    rate_series_name: str = "Target mean",
    show: bool = True,
) -> Tuple[pd.DataFrame, go.Figure]:

    df = data.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col])

    # месяц = начало месяца
    df["month"] = df[date_col].dt.to_period("M").dt.to_timestamp()

    # count = число строк; rate = среднее target (доля)
    df_agg = (
        df.groupby("month", as_index=False)
        .agg(
            count=(target_col, "size"),
            # sum=(target_col, "sum"),
            above_zero=(target_col, lambda s: (s > 0).sum()),
            # rate=(target_col, "mean"),
            rate=(target_col, lambda s: round(pd.to_numeric(s, errors="coerce").mean(), 2)),
        )
        .sort_values("month")
    )

    bar_text = df_agg["count"]
    line_text = [
        f"{int(round(r))}" if pd.notna(r) else "" for r in df_agg["rate"]
    ]

    ymax = float(df_agg["count"].max() or 0)
    pad = ymax * 0.1  # запас сверху
    y2max = float(df_agg["rate"].max() or 0)
    pad2 = y2max * 0.25  # запас сверху

    fig = go.Figure()

    # БАРЫ с подписями сверху
    fig.add_bar(
        x=df_agg["month"],
        y=df_agg["count"],
        name=count_series_name,
        text=bar_text,
        textposition="outside",  # подписи над столбцами
        textfont=dict(size=12),
        marker=dict(line=dict(width=0)),
    )

    # ЛИНИЯ таргета (правая ось, проценты)
    fig.add_trace(
        go.Scatter(
            x=df_agg["month"],
            y=df_agg["rate"],
            mode="lines+markers+text",
            name=rate_series_name,
            yaxis="y2",
            text=line_text,
            textposition="top center",
            line=dict(width=3, color="#D62728"),
            marker=dict(size=8, line=dict(width=1, color="white")),
        )
    )

    fig.update_layout(
        title=title,
        xaxis=dict(title=None, tickangle=45),
        # верхний запас по оси Y, чтобы видно было подписи «outside»
        yaxis=dict(
            title="Количество контрактов", tickformat=",.0f", range=[0, ymax + pad]
        ),
        yaxis2=dict(
            title="mean(ОД)",
            overlaying="y",
            side="right",
            # tickformat=".0%",
            range=[0, y2max + pad2],
        ),
        bargap=0.25,
        template="plotly_white",
        legend=dict(orientation="h", yanchor="top", y=-0.35, xanchor="right", x=0.5),
        margin=dict(t=70, r=60, b=70, l=70),
    )

    if show:
        fig.show()

    df_agg["mean"] = (df_agg["rate"] ).round(2).astype(
        str
    ) + ""  # переведем в проценты

    return df_agg, fig

In [ ]:
info_sample_try, fig_sample_try = counts_target_ratio_sample(sample, 'report_date', 'target')

In [ ]:
info_sample_try[['month', 'count', 'mean']]

### Таргет по месяцам (аналог уровня таргета в классификации)
Среднее и медиана целевой переменной по месяцам — видно тренд и сезонность.

In [ ]:
def target_by_month(df, date_col, target_cols, agg="mean", figsize=(12, 4), show=True):
    """График агрегата таргета по месяцам (agg: 'mean' или 'median')."""
    if isinstance(target_cols, str):
        target_cols = [target_cols]

    df = df.copy()
    df["_month"] = df[date_col].dt.to_period("M")
    g = df.groupby("_month", observed=True)[target_cols].agg(agg)

    plot_df = g.reset_index()
    plot_df["_month"] = plot_df["_month"].astype(str)

    fig = px.line(
        plot_df,
        x="_month",
        y=target_cols,
        title=f"Таргет по месяцам ({agg})",
        markers=True,
    )

    fig.update_layout(
        width=int(figsize[0] * 80),
        height=int(figsize[1] * 80),
        yaxis_title=agg,
        legend=dict(
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.02,
        ),
        margin=dict(r=120),
    )
    fig.update_xaxes(tickangle=45)

    if show:
        fig.show()

    return g, fig

In [ ]:
g, fig = target_by_month(sample, "report_date", TARGET, agg="mean")

In [ ]:
Logger.current_logger().report_plotly(
    title="002 Таргет по месяцам",
    series="",
    figure=fig,
)

### Распределение таргета
Гистограмма и ящик с усами — асимметрия, выбросы, нули.

In [ ]:
from plotly.subplots import make_subplots

In [ ]:
def _target_grid(n, ncols):
    nrows = (n + ncols - 1) // ncols
    return nrows, ncols


def _default_figsize(ncols_plot, nrows, aspect=1.2):
    return (4 * ncols_plot, max(4, 3 * nrows * aspect))


def target_histogram(
    df,
    target_cols,
    ncols=2,
    log_scale=False,
    bins=30,
    figsize=None,
    show=True,
):
    """Гистограммы по списку таргетов (отдельная фигура для ClearML)."""
    if isinstance(target_cols, str):
        target_cols = [target_cols]

    n = len(target_cols)
    nrows, ncols_grid = _target_grid(n, ncols)

    if figsize is None:
        figsize = _default_figsize(ncols, nrows)

    fig = make_subplots(
        rows=nrows,
        cols=ncols_grid,
        subplot_titles=list(target_cols),
        horizontal_spacing=0.1,
        vertical_spacing=0.15 if nrows > 1 else 0.12,
    )

    for i, col in enumerate(target_cols):
        r = i // ncols + 1
        c = i % ncols + 1

        x = df[col].dropna().astype(float)
        x_hist = np.log1p(x) if log_scale and x.max() > 0 else x

        counts, edges = np.histogram(x_hist, bins=bins)
        centers = (edges[:-1] + edges[1:]) / 2
        widths = np.diff(edges)

        fig.add_trace(
            go.Bar(
                x=centers,
                y=counts,
                width=widths,
                marker=dict(line=dict(color="black", width=0.5)),
                opacity=0.7,
                showlegend=False,
                name=col,
                hovertemplate="значение: %{x:.4g}<br>число: %{y}<extra></extra>",
            ),
            row=r,
            col=c,
        )

    fig.update_layout(
        width=int(figsize[0] * 80),
        height=int(figsize[1] * 80),
        autosize=False,
        title_text="Гистограмма таргета",
        margin=dict(l=50, r=30, t=70, b=50),
        bargap=0,
    )

    if show:
        fig.show()
    return fig

In [ ]:
fig_hist = target_histogram(sample, [TARGET], log_scale=True, figsize=(10, 5), show=True)

In [ ]:
Logger.current_logger().report_plotly(
    title="003 Распределение таргета",
    series="",
    figure=fig_hist,
)

In [ ]:
def target_boxplot(
    df,
    target_cols,
    ncols=2,
    figsize=None,
    show=True,
):
    if isinstance(target_cols, str):
        target_cols = [target_cols]
    n = len(target_cols)
    nrows, ncols_grid = _target_grid(n, ncols)
    if figsize is None:
        figsize = (3 * ncols_grid, max(3.5, 3 * nrows))
    fig, axes = plt.subplots(nrows, ncols_grid, figsize=figsize, squeeze=False)
    for i, col in enumerate(target_cols):
        r, c = i // ncols, i % ncols
        x = df[col].dropna().astype(float)
        axes[r, c].boxplot(x, vert=False)
        axes[r, c].set_title(f"{col} (box)")
        axes[r, c].set_xlabel(col)
    for j in range(n, nrows * ncols_grid):
        axes.flat[j].set_visible(False)
    fig.suptitle("Boxplot таргета", y=1.02)
    plt.tight_layout()
    if show:
        plt.show()
    return fig

In [ ]:
fig_box = target_boxplot(sample, [TARGET], ncols=1, figsize=(8, 5), show=True)

In [ ]:
Logger.current_logger().report_matplotlib_figure(
    title="004 Boxplot таргет",
    series="",
    figure=fig_box,
    iteration=0,
)

### Сводка по таргетам: среднее, медиана, std, доля нулей
Таблица — общая и при необходимости по месяцам.

In [ ]:
def target_summary(df, target_cols, by_month=False, date_col=None):
    """Таблица: count, mean, median, std, min, max, доля нулей. by_month=True — то же по месяцам."""
    df = df.copy()
    df[target_cols] = df[target_cols].astype(float)
    if by_month and date_col:   
        df["_month"] = df[date_col].dt.to_period("M")
        return df.groupby("_month")[target_cols].agg(
            ["count", "mean", "median", "std", "min", "max"]
        ).round(2)
    res = df[target_cols].agg(["count", "mean", "median", "std", "min", "max"]).T
    res["pct_zero"] = (df[target_cols] == 0).mean() * 100
    return res.round(2)

In [ ]:
target_summary(sample, [TARGET])

In [ ]:
tbl = target_summary(sample, [TARGET], by_month=True, date_col='report_date')

In [ ]:
tbl = tbl.T.droplevel(0).rename_axis("metric").reset_index()
tbl.columns.name = None
tbl.columns = [str(c) for c in tbl.columns]
tbl

In [ ]:
Logger.current_logger().report_table(
    title="005 Размер выбороки/статистика по таргету",
    series=" ",
    table_plot=tbl
)

In [ ]:
tbl.T.to_excel('regression_rep/check.xlsx')

In [ ]:
sample['report_date'].unique()

## 5. Собираем ВДР и фичи вместе

In [ ]:
merged_df = sample.copy() 

In [ ]:
merged_df = merged_df.drop(['current_limit', 'contract_src_id'], axis=1)

In [ ]:
%%time
import gc

MISSING_THRESHOLD = 0.8

for path, filename in tqdm.tqdm(zip(feature_paths, filenames), total=len(feature_paths)):
    feat_df = pd.read_parquet(path)

    merge_key = [c for c in PRIMARY_KEY if c in feat_df.columns]
    feat_df = feat_df.drop_duplicates(subset=merge_key, keep='first')

    missing_df = preprocessing.analyze_missing_values(
        feat_df, missing_threshold=MISSING_THRESHOLD
    )
    if missing_df is not None and not missing_df.empty:
        missing_df.to_excel(f'{REPORTS_FOLDER}/{filename}_missing.xlsx')
        feat_df = preprocessing.remove_columns_by_missing(feat_df, missing_df, 0.8)

    merged_df = merged_df.merge(feat_df, on=merge_key, how='left', copy=False)

    del feat_df, missing_df
    gc.collect()

In [ ]:
merged_df.info()
merged_df.shape

In [ ]:
merged_df.drop(columns=TECH_COLS, inplace=True, errors="ignore")

In [ ]:
display(merged_df.columns[:10])
all_features = merged_df.columns[4:] # Список всех доступных фичей

In [ ]:
gc.collect()

### Проверка

In [ ]:
display(merged_df.shape)
display(merged_df.head(3))

## 6. Отбор фичей (однофакторный анализ)


1. Убираем фичи с большой долей пропущенных значений
2. Убираем константные фичи
3. Убираем нестабильные фичи (`PSI > threshold`)
4. Разбиение на сегменты: TRAIN, TEST, OOT
5. Грубый отбор фичей (Оставляем ТОП-N фичей)

### 6.1 Убираем фичи с большой долей пропусков

In [ ]:
MISSING_THRESHOLD = 0.8  # Доля пропусков не более чем

missing_df = preprocessing.analyze_missing_values(merged_df, missing_threshold=MISSING_THRESHOLD)
missing_df.to_excel(f'{REPORTS_FOLDER}/missing_percentage.xlsx')

In [ ]:
merged_df = preprocessing.remove_columns_by_missing(merged_df, missing_df, 0.8)

In [ ]:
merged_df.shape

### 6.2 Убираем фичи с константными значениями (низкая дисперсия данных)

In [ ]:
res = preprocessing.find_constant_features(merged_df, threshold=0.95, verbose=False)
res['constant']

In [ ]:
merged_df.drop(res['constant'], axis = 1, inplace =True)

In [ ]:
merged_df.shape

### 6.3 Отбор по PSI

In [ ]:
features = list(merged_df.drop(['report_date', 'client_did', 'contract_uid', 'target_50', 'target_75', 'target_25', TARGET, 'year_month'], axis=1).columns)

In [ ]:
features

In [ ]:
%%time
psi_result = preprocessing.calculate_feat_psi(merged_df[list(features) + ['year_month']], features, date_col='year_month')

In [ ]:
psi_result.head(20)

In [ ]:
# Фильтр 1: глобальное ограничение PSI на всей выборке
PSI_THRESHOLD = 0.15 # Убираем фичи, с PSI > PSI_THRESHOLD
# Фильтр 2: Ограничение тренда
# Если есть линейный тренд на рост PSI, то максимальный тренд PSI не более PSI_TREND
PSI_TREND = 0.1
# PSI_TREND = 0.1

In [ ]:
to_del_all = []
for col in psi_result.columns:
    max_psi = psi_result[col].max()
    lrgs = linregress(range(len(psi_result[col])), psi_result[col])
    if ((lrgs.slope > 0) & (lrgs.pvalue < 0.05) & (psi_result[col].max() > PSI_TREND)) or psi_result[col].max() >= PSI_THRESHOLD:
        to_del_all.append(col)

In [ ]:
psi_result[to_del_all].to_excel(f'{REPORTS_FOLDER}/psi_to_delete.xlsx')

In [ ]:
len(to_del_all), merged_df.shape

In [ ]:
psi_result[ [   
    'current_limit',
    'contract_init_amt',
    'current_debt',
    # 'current_amt',
    'card_age_months',
    'current_interest_rate',
    'prol_prc_rt',
    'prc_rt_diff',
    'contract_amt_diff_init',
    'available_limit_by_debt',
    'util_current',
    'debt_daily_max_90d',
    'debt_daily_max_180d',
    'util_daily_max_90d',
    'util_daily_max_180d',
    'util_monthavg_avg_3m',
    'util_monthavg_avg_6m',
    'util_monthavg_avg_12m',
    'days_since_last_transaction',
	'risk_grade_last']]

In [ ]:
merged_df.drop(to_del_all, axis=1, inplace=True)

In [ ]:
merged_df.shape

In [ ]:
merged_df['target'] = merged_df['target'].astype(float)

### 6.4 Разбиение на сегменты: Train test split

Сегменты:
* `train` - Обучающая часть
* `test` - тестовая часть
* `oot` - в зависимости от типа валидации
* `other` - дубли, некорректные записи и тд

In [ ]:
df = merged_df[merged_df['report_date'] < '2026.02.01']
oot = merged_df[merged_df['report_date'] >= '2026.02.01']
train = df[df['report_date'] < '2026.01.01']
test = df[df['report_date'] >= '2026.01.01']

In [ ]:
train.shape, test.shape, oot.shape

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from typing import Tuple, Optional, Sequence, List, Any

def counts_target_ratio(
    train: pd.DataFrame,
    test: pd.DataFrame,
    oot: pd.DataFrame = None,
    date_col: str = "month",
    target_col: str = "target",
    id_col: str = None,
    title: str = "Размеры выборок и уровень таргета по месяцам",
) -> Tuple[pd.DataFrame, go.Figure]:
    """
    Строит график с размерами выборок train/test/oot и уровнем таргета по месяцам.

    Параметры:
    ----------
    train : pd.DataFrame
        Обучающая выборка
    test : pd.DataFrame
        Тестовая выборка
    oot : pd.DataFrame, optional
        OOT выборка (по умолчанию None)
    date_col : str, optional
        Название колонки с датой (по умолчанию "month")
    target_col : str, optional
        Название колонки с таргетом (по умолчанию "target")
    id_col : str, optional
        Название колонки с ID для подсчета уникальных значений (по умолчанию None)
    title : str, optional
        Заголовок графика (по умолчанию "Размеры выборок и уровень таргета по месяцам")

    Возвращает:
    -----------
    Tuple[pd.DataFrame, go.Figure]
        Агрегированные данные и Plotly фигура
    """

    def _prep(df: pd.DataFrame):
        if df is None or len(df) == 0:
            return pd.DataFrame(columns=["month", "cnt", "rr"])
        tmp = df.copy()
        tmp[date_col] = pd.to_datetime(tmp[date_col])
        tmp["month"] = tmp[date_col].dt.to_period("M").dt.to_timestamp()
        if id_col:
            g = tmp.groupby("month", as_index=False).agg(
                cnt=(id_col, pd.Series.nunique), rr=(target_col, "mean")
            )
        else:
            g = tmp.groupby("month", as_index=False).agg(
                cnt=(target_col, "size"), rr=(target_col, "mean")
            )
        return g

    g_train = _prep(train).rename(columns={"cnt": "train_cnt", "rr": "rr_train"})
    g_test = _prep(test).rename(columns={"cnt": "test_cnt", "rr": "rr_test"})
    g_oot = _prep(oot).rename(columns={"cnt": "oot_cnt", "rr": "rr_oot"})

    # полная сетка месяцев
    months = pd.DataFrame(
        {
            "month": sorted(
                set(g_train["month"]) | set(g_test["month"]) | set(g_oot["month"])
            )
        }
    )
    df = (
        months.merge(g_train, on="month", how="left")
        .merge(g_test, on="month", how="left")
        .merge(g_oot, on="month", how="left")
    )

    df = df[
        ["month", "train_cnt", "test_cnt", "oot_cnt", "rr_train", "rr_test", "rr_oot"]
    ]

    # NaN → 0 только для счетчиков
    for c in ["train_cnt", "test_cnt", "oot_cnt"]:
        if c in df:
            df[c] = df[c].fillna(0).astype(int)

    # --- маски для линий (None, где счетчик = 0) ---
    rr_train_y = [
        v  if (pd.notna(v) and cnt > 0) else None
        for v, cnt in zip(df.get("rr_train"), df.get("train_cnt", 0))
    ]
    rr_test_y = [
        v  if (pd.notna(v) and cnt > 0) else None
        for v, cnt in zip(df.get("rr_test"), df.get("test_cnt", 0))
    ]
    rr_oot_y = [
        v  if (pd.notna(v) and cnt > 0) else None
        for v, cnt in zip(df.get("rr_oot"), df.get("oot_cnt", 0))
    ]

    fig = go.Figure()

    # stacked bars: train + test (+ oot где есть)
    fig.add_bar(x=df["month"], y=df["train_cnt"], name="train_cnt")
    fig.add_bar(x=df["month"], y=df["test_cnt"], name="test_cnt")
    fig.add_bar(x=df["month"], y=df["oot_cnt"], name="OOT_cnt")

    # lines on secondary axis, masked by nonzero counts
    fig.add_trace(
        go.Scatter(
            x=df["month"],
            y=rr_train_y,
            mode="lines+markers",
            name="RR Train",
            yaxis="y2",
            line=dict(color="#17BECF"),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=df["month"],
            y=rr_test_y,
            mode="lines+markers",
            name="RR Test",
            yaxis="y2",
            line=dict(color="#FFD200"),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=df["month"],
            y=rr_oot_y,
            mode="lines+markers",
            name="RR OOT",
            yaxis="y2",
            line=dict(color="#7B61FF"),
        )
    )

    fig.update_layout(
        barmode="stack",
        title=title,
        xaxis=dict(title="Месяц"),
        yaxis=dict(title="Количество клиентов", tickformat=",.0f"),
        yaxis2=dict(title="RR, %", overlaying="y", side="right"),
        legend=dict(orientation="h", yanchor="top", y=-0.35, xanchor="center", x=0.5),
        template="plotly_white",
    )

    # показать и вернуть
    fig.show()

    df = df.fillna(0)
    df["rr_train"] = (df["rr_train"] ).round(2).astype(
        str
    )   # переведем в проценты
    df["rr_test"] = (df["rr_test"] ).round(2).astype(
        str
    )   # переведем в проценты
    df["rr_oot"] = (df["rr_oot"] ).round(2).astype(
        str
    )   # переведем в проценты
    df = df.replace("0.0", "---")
    df = df.replace(0, "---")
    return df, fig

In [ ]:
info_df, fig_df = counts_target_ratio(train, test, oot, TIME_COL, TARGET)
fig_df.update_layout(
    title="Размеры выборок и средний ОД по месяцам",
    yaxis2=dict(title="ОД", overlaying="y", side="right"),
)
for tr in fig_df.data:
    if tr.name and tr.name.startswith("RR "):
        tr.name = tr.name.replace("RR ", "ОД ")

info_df = info_df.rename(columns={
    "rr_train": "util_train",
    "rr_test": "util_test",
    "rr_oot": "util_oot",
})
fig_df

In [ ]:
Logger.current_logger().report_plotly(
    title="006 Размеры выборок и средний ОД по месяцам",
    series="",
    figure=fig_df)

In [ ]:
info_df = info_df.rename(columns={
    "util_train": "mean(target) train",
    "util_test": "mean(target) test",
    "util_oot": "mean(target) oot",
})
info_df

In [ ]:
info_df_to_log = info_df.copy()
info_df_to_log["month"] = pd.to_datetime(info_df_to_log["month"]).dt.strftime("%Y-%m-%d")
Logger.current_logger().report_table(
    title="007 Размеры выборок и средний таргет по месяцам",
    series="",
    table_plot=info_df_to_log
)
info_df

In [ ]:
# Проверка на потерянные строки 
assert train[PRIMARY_KEY].shape[0] +\
        test[PRIMARY_KEY].shape[0] +\
         oot[PRIMARY_KEY].shape[0] == sample.dropna(subset=[TARGET]).shape[0]

## 7. Предварительный отбор фичей

In [ ]:
# Доработать
categorical_features = list(merged_df
                            .drop(columns=PRIMARY_KEY+['year_month','target_25','target_50','target_75',TARGET])
                            .select_dtypes(['category', 'object'])
                            .columns)
numerical_features = list(merged_df
                          .drop(columns=categorical_features + PRIMARY_KEY + ['year_month','target_25','target_50','target_75',TARGET])
                          .columns)

print(f'Categorical: {len(categorical_features)}')
print(f'Numerical: {len(numerical_features)}')

In [ ]:
features = categorical_features + numerical_features

In [ ]:
features

## Tweedie

In [ ]:
def check_working_model(train, train_pred, test, test_pred, oot, oot_pred):
    y = train["target"].astype(float).values
    y_const = np.full_like(y, y.mean())
    print("MAE const:", mean_absolute_error(y, y_const))
    print("R2 model train :", r2_score(y, train_pred))
    print("R2 model test  :", r2_score(test["target"].astype(float).values, test_pred))
    print("R2 model oot   :", r2_score(oot["target"].astype(float).values, oot_pred))

In [ ]:
# LGBM_KW_TWEEDIE = dict(
#     objective="tweedie",
#     tweedie_variance_power=1.5,
# )

In [ ]:
def select_by_importance(data, target_col, eval_data, max_features=None, importance_threshold=None, random_state=42):
    """
    Важность из LGBMRegressor. Возвращает (selected, imp, model) — model для evaluate_model.
    """
    X = data.drop(columns=[target_col])
    y = data[target_col]
    eval_set = [(eval_data.drop(columns=[target_col]), eval_data[target_col])]
    X_ = X.select_dtypes(include=[np.number]).copy()
    eval_align = [(X_val[X_.columns], y_val) for X_val, y_val in eval_set]
    m = lgb.LGBMRegressor(random_state=random_state, n_estimators=100, max_depth=4, verbosity=-1)
    m.fit(X_, y, eval_set=eval_align)
    imp = pd.Series(m.feature_importances_, index=X_.columns).sort_values(ascending=False)
    selected = imp[imp >= importance_threshold].index.tolist() if importance_threshold is not None else imp.index.tolist()
    if max_features is not None:
        selected = imp.head(max_features).index.tolist()
    return selected, imp, m


def evaluate_model(model, train_df, test_df, oot_df, target_col):
    """
    Сводка по модели: Train / Test / OOT. Возвращает DataFrame с колонками:
    Target, AVG target, AVG pred, Dataset, MAPE, MAE, MAE 95%, RMSE, Non Zero.
    MAE 95% — MAE по 95% данных без 5% наибольших значений таргета y (выбросы по y).
    """
    feat = getattr(model, "feature_name_", None) or getattr(model, "feature_names_in_", None)
    if feat is None:
        feat = [c for c in train_df.columns if c != target_col]
    rows = []
    for name, df in [("Train", train_df), ("Test", test_df), ("OOT", oot_df)]:
        X = df[feat] if all(c in df.columns for c in feat) else df.drop(columns=[target_col])
        y = df[target_col].values
        pred = model.predict(X)
        nz = (y != 0)
        mape = np.mean(np.abs((y - pred)[nz]) / y[nz]) if nz.sum() > 0 else np.nan
        mae = np.mean(np.abs(y - pred))
        thresh_y = np.percentile(y, 95)
        mask = y <= thresh_y
        mae_95 = np.mean(np.abs(y[mask] - pred[mask])) if mask.sum() > 0 else np.nan
        rmse = np.sqrt(np.mean((y - pred) ** 2))
        rows.append({
            "Target": target_col,
            "AVG target": y.mean(),
            "AVG pred": pred.mean(),
            "Dataset": name,
            # "MAPE": mape,
            "MAE": mae,
            "MAE 95%": mae_95,
            "RMSE": rmse,
            "Non Zero": int(nz.sum()),
        })
    return pd.DataFrame(rows)

In [ ]:
def metric_vs_n_features(data, target_col, eval_data, imp, step=30, metric="rmse", random_state=42, threshold=0.005):
    """
    Перебор числа признаков по важности: от 20 с шагом step до n_all. imp — результат select_by_importance.
    threshold: порог — берётся наименьшее n, при котором метрика <= минимум * (1 + threshold). Например 0.005 = 0.5%.
    Если None — просто argmin. Возвращает (report_df, best_n, selected, model).
    """
    MIN_FEATURES = 20
    n_all = len(imp)
    start = min(MIN_FEATURES, n_all)
    n_features_list = list(range(start, n_all + 1, step))
    if n_all not in n_features_list:
        n_features_list.append(n_all)
    n_features_list = sorted(set(n_features_list))

    X_ev = eval_data[imp.index.tolist()]
    y_ev = eval_data[target_col].values
    fn = mean_squared_error if metric == "rmse" else mean_absolute_error

    rows = []
    for n in n_features_list:
        cols = imp.head(n).index.tolist()
        X_tr = data[cols]
        y_tr = data[target_col]
        m = lgb.LGBMRegressor(random_state=random_state, n_estimators=100, max_depth=4, verbosity=-1)
        m.fit(X_tr, y_tr, eval_set=[(X_ev[cols], y_ev)])
        pred = m.predict(X_ev[cols])
        val = np.sqrt(fn(y_ev, pred)) if metric == "rmse" else fn(y_ev, pred)
        rows.append({"n_features": n, "metric_value": val})

    report_df = pd.DataFrame(rows)
    best_metric = report_df["metric_value"].min()
    if threshold is not None:
        metric_limit = best_metric * (1 + threshold)
        candidates = report_df[report_df["metric_value"] <= metric_limit].sort_values("n_features")
        best_n = int(candidates.iloc[0]["n_features"]) if len(candidates) > 0 else int(report_df.loc[report_df["metric_value"].idxmin(), "n_features"])
    else:
        best_n = int(report_df.loc[report_df["metric_value"].idxmin(), "n_features"])
    selected = imp.head(best_n).index.tolist()
    model = lgb.LGBMRegressor(random_state=random_state, n_estimators=100, max_depth=4, verbosity=-1)
    model.fit(data[selected], data[target_col], eval_set=[(eval_data[selected], eval_data[target_col])])

    plt.figure(figsize=(8, 4))
    plt.plot(report_df["n_features"], report_df["metric_value"], marker="o")
    plt.axvline(best_n, color="gray", linestyle="--", label=f"best n={best_n}")
    plt.xlabel("Число признаков")
    plt.ylabel(metric.upper())
    plt.title(f"Метрика на eval от n для {target_col}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    return report_df, best_n, selected, model

In [ ]:
def tune_hyperparams(data, target_col, eval_data, max_evals=30, metric="rmse", random_state=42):
    """
    Подбор гиперпараметров LGBM через Hyperopt (TPE). 
    Возвращает (best_model, best_params).
    """
    X = data.drop(columns=[target_col])
    y = data[target_col]
    X_ev = eval_data.drop(columns=[target_col])
    y_ev = eval_data[target_col].values
    fn = mean_squared_error if metric == "rmse" else mean_absolute_error

    space = {
        "n_estimators": hp.quniform("n_estimators", 50, 300, 25),
        "max_depth": hp.quniform("max_depth", 3, 10, 1),
        "learning_rate": hp.loguniform("learning_rate", np.log(0.02), np.log(0.3)),
        "num_leaves": hp.quniform("num_leaves", 15, 127, 10),
        "min_child_samples": hp.quniform("min_child_samples", 5, 50, 5)
    }

    def objective(params):
        p = {
            "n_estimators": int(params["n_estimators"]),
            "max_depth": int(params["max_depth"]),
            "learning_rate": params["learning_rate"],
            "num_leaves": int(params["num_leaves"]),
            "min_child_samples": int(params["min_child_samples"]),
        }
        m = lgb.LGBMRegressor(random_state=random_state, verbosity=-1, **p)
        m.fit(X, y, eval_set=[(X_ev, y_ev)])
        pred = m.predict(X_ev)
        score = np.sqrt(fn(y_ev, pred)) if metric == "rmse" else fn(y_ev, pred)
        return {"loss": score, "status": STATUS_OK}

    trials = Trials()
    best = fmin(
        objective, space, algo=tpe.suggest, max_evals=max_evals, trials=trials,
        rstate=np.random.default_rng(random_state),
    )
    best_params = {
        "n_estimators": int(best["n_estimators"]),
        "max_depth": int(best["max_depth"]),
        "learning_rate": best["learning_rate"],
        "num_leaves": int(best["num_leaves"]),
        "min_child_samples": int(best["min_child_samples"])
    }
    best_model = lgb.LGBMRegressor(random_state=random_state, verbosity=-1, **best_params)
    best_model.fit(X, y, eval_set=[(X_ev, y_ev)])
    return best_model, best_params

In [ ]:
gc.collect()

In [ ]:
%%time
PRESELECTION_TOP_N = 150  # Количество предварительно отобранных фичей

results = {}

for target in [TARGET]:
    # Предварительный отбор фичей
    selected, imp, _ = select_by_importance(train[features + [target]], target, 
                                                eval_data=test[features + [target]], max_features=PRESELECTION_TOP_N)
    # Подбор оптимального количества фичей
    report_n, best_n, selected, model = metric_vs_n_features(train[selected + [target]], target, 
                                                             eval_data=test[selected + [target]], imp=imp[selected], step=10, threshold=0.01)
    # Подбор гиперпараметров
    best_model, best_params = tune_hyperparams(train[selected + [target]], target, test[selected + [target]], max_evals=30)
    report = evaluate_model(best_model, train[selected + [target]], test[selected + [target]], oot[selected + [target]], target)

    results[target] = {
        "selected": selected,
        "model": best_model,
        "evaluate_report": report,
        "best_n": best_n,
        "best_params": best_params,
        "metric_vs_n": report_n,
    }

# Сводная таблица по всем таргетам (Train/Test/OOT)
reports_all = pd.concat([results[t]["evaluate_report"] for t in [TARGET]], ignore_index=True)

In [ ]:
reports_all

In [ ]:
check_working_model(train, results['target']['model'].predict(train[results['target']['selected']]),
                   test, results['target']['model'].predict(test[results['target']['selected']]),
                   oot, results['target']['model'].predict(oot[results['target']['selected']]))

In [ ]:
results['target']

In [ ]:
# === Срез: нулевая утилизация и сегмент y>0 ===

ZERO_EPS = 1e-9          # фактический ноль (учёт float)
THRESHOLDS = [0.03, 0.05, 0.10, 0.15, 0.20]  # пороги pred «низкая утилизация»

model = results[TARGET]["model"]
feat = results[TARGET]["selected"]


def _zero_slice_one(df, name, thresholds=THRESHOLDS):
    y = df[TARGET].astype(float).values
    pred = model.predict(df[feat])
    is_zero = y <= ZERO_EPS
    n = len(y)
    n_zero = int(is_zero.sum())
    nz = ~is_zero

    rows = []
    for t in thresholds:
        pred_low = pred < t
        tp = int((pred_low & is_zero).sum())
        fp = int((pred_low & ~is_zero).sum())
        fn = int((~pred_low & is_zero).sum())
        tn = int((~pred_low & ~is_zero).sum())
        n_pred_low = int(pred_low.sum())
        precision = tp / n_pred_low if n_pred_low > 0 else np.nan
        recall = tp / n_zero if n_zero > 0 else np.nan
        rows.append({
            "Dataset": name,
            "Порог pred": t,
            "Доля клиентов pred < порог, %": round(100 * n_pred_low / n, 2),
            "Precision (среди pred<порог — доля реальных нулей), %": round(100 * precision, 2) if pd.notna(precision) else np.nan,
            "Recall (доля пойманных нулей), %": round(100 * recall, 2) if pd.notna(recall) else np.nan,
            "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        })

    summary = pd.DataFrame(rows)

    mae_all = mean_absolute_error(y, pred)
    mae_nz = mean_absolute_error(y[nz], pred[nz]) if nz.sum() > 0 else np.nan
    mae_zero = mean_absolute_error(y[is_zero], pred[is_zero]) if n_zero > 0 else np.nan
    r2 = r2_score(y, pred)
    mae_const = mean_absolute_error(y, np.full_like(y, y.mean()))

    metrics = pd.DataFrame([{
        "Dataset": name,
        "N": n,
        "Доля факт. нулей, %": round(100 * n_zero / n, 2),
        "MAE (все)": round(mae_all, 4),
        "MAE (y=0)": round(mae_zero, 4),
        "MAE (y>0)": round(mae_nz, 4),
        "MAE const (baseline)": round(mae_const, 4),
        "R2": round(r2, 4),
        "AVG target": round(y.mean(), 4),
        "AVG pred": round(pred.mean(), 4),
    }])

    return summary, metrics


thresh_tables = []
metric_tables = []
for split_name, split_df in [("Train", train), ("Test", test), ("OOT", oot)]:
    th, mt = _zero_slice_one(split_df, split_name)
    thresh_tables.append(th)
    metric_tables.append(mt)

zero_threshold_report = pd.concat(thresh_tables, ignore_index=True)
zero_metrics_report = pd.concat(metric_tables, ignore_index=True)

print("=== Основные метрики (MAE / R²) ===")
display(zero_metrics_report)

print("\n=== Качество выделения нулевой утилизации по порогу pred ===")
print("(Precision: среди низкого прогноза — сколько реально нулей; Recall: какую долю всех нулей поймали)")
display(zero_threshold_report)

print("\n=== OOT ===")
display(zero_threshold_report[zero_threshold_report["Dataset"] == "OOT"])

# Опционально: лучший порог на OOT по F1 для класса «ноль»
oot_rows = zero_threshold_report[zero_threshold_report["Dataset"] == "OOT"].copy()
prec = oot_rows["TP"] / (oot_rows["TP"] + oot_rows["FP"]).replace(0, np.nan)
rec = oot_rows["TP"] / (oot_rows["TP"] + oot_rows["FN"]).replace(0, np.nan)
oot_rows["F1 zero, %"] = (200 * prec * rec / (prec + rec)).round(2)
best_oot = oot_rows.loc[oot_rows["F1 zero, %"].idxmax()]
print(
  f"\nНа OOT максимальный F1 по классу «ноль»: порог pred = {best_oot['Порог pred']}, "
  f"F1 = {best_oot['F1 zero, %']}%, "
  f"Recall = {best_oot['Recall (доля пойманных нулей), %']}%, "
  f"Precision = {best_oot['Precision (среди pred<порог — доля реальных нулей), %']}%"
)

In [ ]:
ZERO_EPS = 1e-9          # фактический ноль (учёт float)
THRESHOLD_START = 0.01
THRESHOLD_END = 0.20
THRESHOLD_STEP = 0.01
THRESHOLDS = np.round(
    np.arange(THRESHOLD_START, THRESHOLD_END + THRESHOLD_STEP / 2, THRESHOLD_STEP), 2
)  # пороги pred «низкая утилизация», шаг 0.01

model = results[TARGET]["model"]
feat = results[TARGET]["selected"]


def _zero_slice_one(df, name, thresholds=THRESHOLDS):
    y = df[TARGET].astype(float).values
    pred = model.predict(df[feat])
    is_zero = y <= ZERO_EPS
    n = len(y)
    n_zero = int(is_zero.sum())
    nz = ~is_zero

    rows = []
    for t in thresholds:
        pred_low = pred < t
        tp = int((pred_low & is_zero).sum())
        fp = int((pred_low & ~is_zero).sum())
        fn = int((~pred_low & is_zero).sum())
        tn = int((~pred_low & ~is_zero).sum())
        n_pred_low = int(pred_low.sum())
        precision = tp / n_pred_low if n_pred_low > 0 else np.nan
        recall = tp / n_zero if n_zero > 0 else np.nan
        if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0:
            f1 = 2 * precision * recall / (precision + recall)
        else:
            f1 = np.nan
        rows.append({
            "Dataset": name,
            "Порог pred": t,
            "Доля клиентов pred < порог, %": round(100 * n_pred_low / n, 2),
            "Precision (среди pred<порог — доля реальных нулей), %": round(100 * precision, 2) if pd.notna(precision) else np.nan,
            "Recall (доля пойманных нулей), %": round(100 * recall, 2) if pd.notna(recall) else np.nan,
            "F1 (класс «ноль»), %": round(100 * f1, 2) if pd.notna(f1) else np.nan,
            "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        })

    summary = pd.DataFrame(rows)

    mae_all = mean_absolute_error(y, pred)
    mae_nz = mean_absolute_error(y[nz], pred[nz]) if nz.sum() > 0 else np.nan
    mae_zero = mean_absolute_error(y[is_zero], pred[is_zero]) if n_zero > 0 else np.nan
    r2 = r2_score(y, pred)
    mae_const = mean_absolute_error(y, np.full_like(y, y.mean()))

    metrics = pd.DataFrame([{
        "Dataset": name,
        "N": n,
        "Доля факт. нулей, %": round(100 * n_zero / n, 2),
        "MAE (все)": round(mae_all, 4),
        "MAE (y=0)": round(mae_zero, 4),
        "MAE (y>0)": round(mae_nz, 4),
        "MAE const (baseline)": round(mae_const, 4),
        "R2": round(r2, 4),
        "AVG target": round(y.mean(), 4),
        "AVG pred": round(pred.mean(), 4),
    }])

    return summary, metrics


thresh_tables = []
metric_tables = []
for split_name, split_df in [("Train", train), ("Test", test), ("OOT", oot)]:
    th, mt = _zero_slice_one(split_df, split_name)
    thresh_tables.append(th)
    metric_tables.append(mt)

zero_threshold_report = pd.concat(thresh_tables, ignore_index=True)
zero_metrics_report = pd.concat(metric_tables, ignore_index=True)

print("=== Основные метрики (MAE / R²) ===")
display(zero_metrics_report)

print("\n=== Качество выделения нулевой утилизации по порогу pred ===")
print("(Precision: среди низкого прогноза — сколько реально нулей; Recall: какую долю всех нулей поймали)")
display(zero_threshold_report)

print("\n=== OOT: только пороги ===")
display(zero_threshold_report[zero_threshold_report["Dataset"] == "OOT"])

# Лучший порог на OOT по F1
oot_rows = zero_threshold_report[zero_threshold_report["Dataset"] == "OOT"].copy()
f1_col = "F1 (класс «ноль»), %"
best_oot = oot_rows.loc[oot_rows[f1_col].idxmax()]
print(
    f"\nНа OOT максимальный F1 по классу «ноль»: порог pred = {best_oot['Порог pred']}, "
    f"F1 = {best_oot[f1_col]}%, "
    f"Recall = {best_oot['Recall (доля пойманных нулей), %']}%, "
    f"Precision = {best_oot['Precision (среди pred<порог — доля реальных нулей), %']}%"
)

In [ ]:
zero_metrics_report.to_excel('metrics.xlsx')
zero_threshold_report.to_excel('threshold.xlsx')

In [ ]:
best_features = results[target]['selected']
features_sql = ','.join(f"'{f}'" for f in best_features)

engine_sb_dirkaim = oracle.create_engine_cdw(USERNAME_SB_DIRKAIM, PASSWORD_SB_DIRKAIM)
with engine_sb_dirkaim.connect() as conn:
    description_df =  pd.read_sql(f"""select * from SB_DIRKAIM.FEATURE_REGISTRY where f_name in ({features_sql})""", conn)
description_df = description_df[['f_name', 'f_desc', 'proc']]

In [ ]:
description_df

In [ ]:
# имена фичей в том же порядке, что у модели
names = model.feature_name_ or model.feature_names_in_ or feat

imp = pd.Series(model.feature_importances_, index=names).sort_values(ascending=False)
imp_df = imp.reset_index()
imp_df.columns = ["feature", "importance"]

display(imp_df)

In [ ]:
df_descr = imp_df.merge(description_df, left_on='feature', right_on='f_name', how='left')[['feature', 'f_desc', 'proc', 'importance']].sort_values(by='importance', ascending=False)

In [ ]:
# df_descr.to_excel('imp_descr.xlsx')

In [ ]:
def create_df(df, target, pred_col, y_pred, segment_name):
    result_df = df[PRIMARY_KEY+[target]].copy()
    result_df[pred_col] = y_pred
    result_df['segment'] = segment_name

    return result_df

In [ ]:
os.makedirs(f"{PATH_MODELS_DATA}/{target}/", exist_ok=True)
best_features = results[target]['selected']
train_df = create_df(train, target, f'{target}_pred', results[target]['model'].predict(train[best_features]), 'train')
test_df = create_df(test, target, f'{target}_pred', results[target]['model'].predict(test[best_features]), 'test')
oot_df = create_df(oot, target, f'{target}_pred', results[target]['model'].predict(oot[best_features]), 'oot')

scored_df = pd.concat([train_df, test_df, oot_df], ignore_index=True)

# сохраняем train, test, oot выборки после отбора фичей и проскоренную выборку
train[PRIMARY_KEY + best_features + [target]].to_parquet(f"{PATH_MODELS_DATA}/{target}/sample_train.parquet")
test[PRIMARY_KEY + best_features + [target]].to_parquet(f"{PATH_MODELS_DATA}/{target}/sample_test.parquet")
oot[PRIMARY_KEY + best_features + [target]].to_parquet(f"{PATH_MODELS_DATA}/{target}/sample_oot.parquet")
scored_df.to_parquet(f"{PATH_MODELS_DATA}/{target}/scored_df.parquet")

In [ ]:
target = TARGET
pred_col = f"{target}_pred"
N_PER_SEGMENT = 30   # строк на каждый segment
SEED = 42

demo_parts = []
for seg in ["train", "test", "oot"]:
    part = scored_df.loc[scored_df["segment"] == seg]
    n = min(N_PER_SEGMENT, len(part))
    if n > 0:
        demo_parts.append(part.sample(n=n, random_state=SEED))

scored_df_demo = pd.concat(demo_parts, ignore_index=True)

if "error" not in scored_df_demo.columns:
    scored_df_demo["error"] = scored_df_demo[target].astype(float) - scored_df_demo[pred_col]

cols_demo = ["segment"] + PRIMARY_KEY + [target, pred_col, "error"]
scored_df_demo_view = scored_df_demo[cols_demo].round(4)

print(
    f"Случайная выборка: {len(scored_df_demo_view)} строк "
    f"({scored_df_demo_view['segment'].value_counts().to_dict()})"
)
display(scored_df_demo_view)

demo_path = f"scored_df_demo_sample.xlsx"
scored_df_demo_view.to_excel(demo_path, index=False)
print(f"Сохранено: {demo_path}")

## HTML отчет

In [ ]:
# Поля, заполняемые вручную:
TITLE = "Модель прогноза средней ежедневной задолженности за 6‑й месяц" # Название модели
DESCRIPTION = "Предсказание средней ежедневной задолженности за 6‑й месяц: дни 151–180 после даты скоринга." # Формализованное описание таргета 
TRAIN_TEST_PERIOD = "Сентябрь 2025 - январь 2026"
OOT_PERIOD = "Февраль 2026"
MODE = "offline model" # Режим работы (online/offline)
MODEL_TYPE = "LighGBM"
POPULATION_TEXT = "Клиенты имеющие КК" # Описание выборки (напр. Оформили КК или получили СМС с предложением)

In [ ]:
# Техническое имя модели
product = 'ptb_declip' # +- свободное наименование, главное отразить бизнес-процесс или бизнес-задачу, которую решает модель
monthYear = datetime.datetime.now().strftime('%m%y')

MODEL_NAME = f"{ML_TYPE}_{product}_{monthYear}_{SUBTASK}"

print(MODEL_NAME)

In [ ]:
from toolbox.modeling_utils import report

In [ ]:
model = results[TARGET]["model"]
feat = results[TARGET]["selected"]
scored_for_psi = pd.concat(
    [
        train.assign(pred=model.predict(train[feat])),
        test.assign(pred=model.predict(test[feat])),
        oot.assign(pred=model.predict(oot[feat])),
    ],
    ignore_index=True,
)
psi_scores = preprocessing.calculate_feat_psi(
    scored_for_psi[["pred", "year_month"]],
    ["pred"],
    date_col="year_month",
)
_, psi_score_fig = report.psi_plotly(psi_scores.T, "PSI score")

In [ ]:
_, psi_fig = report.psi_plotly(psi_result[feat].T)

In [ ]:
def regression_metrics_summary(model, train_df, test_df, oot_df, target_col, n_feat=None):
    """
    Сводная таблица качества Train / Test / OOT.
    Аналог блока с ROC / Gini / F1 в классификации.
    """
    feat = _model_features(model, train_df, target_col)
    if n_feat is None:
        n_feat = len(feat)

    rows = []
    for name, df in [("Train", train_df), ("Test", test_df), ("OOT", oot_df)]:
        y = pd.to_numeric(df[target_col], errors="coerce").values
        pred = model.predict(df[feat])
        nz = y > 0
        z = y == 0
        mae_95_mask = y <= np.percentile(y, 95)

        rows.append({
            "Dataset": name,
            "N_samples": len(df),
            "N_feat": n_feat,
            "avg_target": round(float(np.nanmean(y)), 6),
            "avg_pred": round(float(np.mean(pred)), 6),
            "share_target_gt0": round(float(nz.mean()), 6),
            "MAE": round(mean_absolute_error(y, pred), 6),
            "RMSE": round(mean_squared_error(y, pred) ** 0.5, 6),
            "R2": round(r2_score(y, pred), 6),
            "MAE_95pct": round(mean_absolute_error(y[mae_95_mask], pred[mae_95_mask]), 6),
            "MAE_nonzero": round(mean_absolute_error(y[nz], pred[nz]), 6) if nz.any() else np.nan,
            "MAE_zero": round(mean_absolute_error(y[z], pred[z]), 6) if z.any() else np.nan,
        })

    return pd.DataFrame(rows)

In [ ]:
def regression_metrics_by_month(df, target_col, pred_col, date_col="year_month"):
    """
    Метрики по месяцам — таблица для графика (аналог Gini по месяцам).
    df должен содержать pred (см. add_pred).
    """
    tmp = df[[date_col, target_col, pred_col]].copy()
    tmp[target_col] = pd.to_numeric(tmp[target_col], errors="coerce")
    tmp[pred_col] = pd.to_numeric(tmp[pred_col], errors="coerce")
    tmp[date_col] = pd.to_datetime(tmp[date_col])

    rows = []
    for month, g in tmp.groupby(date_col, sort=True):
        y = g[target_col].values
        p = g[pred_col].values
        rows.append({
            "month": month,
            "cnt": len(g),
            "avg_target": y.mean(),
            "avg_pred": p.mean(),
            "share_gt0": (y > 0).mean(),
            "mae": mean_absolute_error(y, p),
            "rmse": mean_squared_error(y, p) ** 0.5,
            "r2": r2_score(y, p) if len(g) > 1 else np.nan,
        })

    return pd.DataFrame(rows)


def regression_metrics_by_month_plotly(df_month, metric_col="mae", title="Динамика качества по месяцам"):
    """Столбцы cnt + линия метрики (MAE / RMSE / R2) — аналог Gini по месяцам."""
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_bar(x=df_month["month"], y=df_month["cnt"], name="cnt", opacity=0.35)
    fig.add_scatter(
        x=df_month["month"],
        y=df_month[metric_col],
        name=metric_col.upper(),
        mode="lines+markers",
        yaxis="y2",
    )
    fig.update_layout(
        title=title,
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    )
    fig.update_xaxes(title_text="Месяц")
    fig.update_yaxes(title_text="Количество клиентов", secondary_y=False)
    fig.update_yaxes(title_text=metric_col.upper(), secondary_y=True)
    return fig


def regression_avg_target_pred_plotly(df_month, title="Средний таргет vs предсказание по месяцам"):
    """Калибровка уровня во времени: avg_target и avg_pred."""
    fig = go.Figure()
    fig.add_scatter(
        x=df_month["month"], y=df_month["avg_target"],
        name="avg_target", mode="lines+markers",
    )
    fig.add_scatter(
        x=df_month["month"], y=df_month["avg_pred"],
        name="avg_pred", mode="lines+markers",
    )
    fig.update_layout(title=title, template="plotly_white", xaxis_title="Месяц")
    return fig

In [ ]:
def regression_decile_table(df, pred_col, target_col, id_col="client_did", n_bins=10):
    """
    Таблица по децилям скора — аналог decile_target в классификации.
    decile=1 — наименьший pred, decile=10 — наибольший.
    """
    d = df[[id_col, target_col, pred_col]].copy()
    d[target_col] = pd.to_numeric(d[target_col], errors="coerce")
    d[pred_col] = pd.to_numeric(d[pred_col], errors="coerce")
    d = d.dropna(subset=[target_col, pred_col])

    d["decile"] = pd.qcut(
        d[pred_col].rank(method="first"),
        q=n_bins,
        labels=False,
        duplicates="drop",
    ) + 1

    g_mean = d[target_col].mean()
    g_sum = d[target_col].sum()

    agg = (
        d.groupby("decile", as_index=False)
        .agg(
            client_all=(id_col, "count"),
            target_gt0=(target_col, lambda s: (s > 0).sum()),
            sum_target=(target_col, "sum"),
            mean_target=(target_col, "mean"),
            mean_pred=(pred_col, "mean"),
        )
    )
    agg["lift"] = agg["mean_target"] / g_mean if g_mean else np.nan
    agg["share_sum_target"] = agg["sum_target"] / g_sum if g_sum else np.nan
    agg["ratio_gt0"] = agg["target_gt0"] / agg["client_all"]
    return agg

In [ ]:
def regression_decile_lift_plotly(decile_df, title="Лифт по децилям"):
    """Dual-axis: lift (bars) + mean_target / mean_pred (lines). Аналог лифт-графика в классификации."""
    global_mean = decile_df["sum_target"].sum() / decile_df["client_all"].sum()

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_bar(x=decile_df["decile"], y=decile_df["lift"], name="lift", opacity=0.35)
    fig.add_scatter(
        x=decile_df["decile"], y=decile_df["mean_target"],
        name="mean_target", mode="lines+markers", yaxis="y2",
    )
    fig.add_scatter(
        x=decile_df["decile"], y=decile_df["mean_pred"],
        name="mean_pred", mode="lines+markers", yaxis="y2",
    )
    fig.add_hline(
        y=global_mean, line_dash="dash",
        annotation_text="avg target", yref="y2",
    )
    fig.update_layout(
        title=title,
        template="plotly_white",
        xaxis_title="decile",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    )
    fig.update_yaxes(title_text="Lift", secondary_y=False)
    fig.update_yaxes(title_text="Среднее", secondary_y=True)
    return fig

In [ ]:
def regression_calibration_bins(df, pred_col, target_col, n_bins=10):
    """
    Калибровка по бинам скора — аналог PR-curve / калибровки вероятностей.
    """
    d = df[[target_col, pred_col]].copy()
    d[target_col] = pd.to_numeric(d[target_col], errors="coerce")
    d[pred_col] = pd.to_numeric(d[pred_col], errors="coerce")
    d = d.dropna(subset=[target_col, pred_col])

    d["bin"] = pd.qcut(
        d[pred_col].rank(method="first"),
        q=n_bins,
        labels=False,
        duplicates="drop",
    ) + 1

    out = (
        d.groupby("bin", as_index=False)
        .agg(
            n=(target_col, "size"),
            mean_pred=(pred_col, "mean"),
            mean_target=(target_col, "mean"),
        )
    )
    out["bias"] = out["mean_pred"] - out["mean_target"]
    return out


def regression_calibration_plotly(cal_df, title="Калибровка по бинам скора"):
    fig = go.Figure()
    fig.add_scatter(
        x=cal_df["bin"], y=cal_df["mean_target"],
        name="mean_target", mode="lines+markers",
    )
    fig.add_scatter(
        x=cal_df["bin"], y=cal_df["mean_pred"],
        name="mean_pred", mode="lines+markers",
    )
    fig.update_layout(
        title=title,
        template="plotly_white",
        xaxis_title="bin (decile pred)",
    )
    return fig

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _model_features(model, train_df, target_col):
    feat = getattr(model, "feature_name_", None) or getattr(model, "feature_names_in_", None)
    if feat is None:
        feat = [c for c in train_df.columns if c != target_col]
    return list(feat)


def add_pred(df, model, feat, pred_col="pred"):
    """Добавляет колонку pred к датафрейму."""
    out = df.copy()
    out[pred_col] = model.predict(out[feat])
    return out

In [ ]:
model = results[TARGET]["model"]
feat = results[TARGET]["selected"]
pred_col = "pred"
id_col = PRIMARY_KEY[0]

scored_train = add_pred(train, model, feat, pred_col)
scored_test = add_pred(test, model, feat, pred_col)
scored_oot = add_pred(oot, model, feat, pred_col)
scored_all = pd.concat([scored_train, scored_test, scored_oot], ignore_index=True)

# 1. Сводные метрики (таблица)
metrics_summary = regression_metrics_summary(
    model, train, test, oot, TARGET, n_feat=len(feat),
)
metrics_summary

# 2. Динамика по месяцам
df_month = regression_metrics_by_month(scored_all, TARGET, pred_col, date_col="year_month")
fig_month_mae = regression_metrics_by_month_plotly(df_month, metric_col="mae")
fig_month_avg = regression_avg_target_pred_plotly(df_month)

# 3. Децили / лифт
train_decile = regression_decile_table(scored_train, pred_col, TARGET, id_col=id_col)
test_decile = regression_decile_table(scored_test, pred_col, TARGET, id_col=id_col)
oot_decile = regression_decile_table(scored_oot, pred_col, TARGET, id_col=id_col)

fig_lift_train = regression_decile_lift_plotly(train_decile, title="Лифт train")
fig_lift_test = regression_decile_lift_plotly(test_decile, title="Лифт test")
fig_lift_oot = regression_decile_lift_plotly(oot_decile, title="Лифт oot")

# 4. Калибровка по бинам скора
cal_train = regression_calibration_bins(scored_train, pred_col, TARGET)
cal_test = regression_calibration_bins(scored_test, pred_col, TARGET)
cal_oot = regression_calibration_bins(scored_oot, pred_col, TARGET)

fig_cal_train = regression_calibration_plotly(cal_train, title="Калибровка train")
fig_cal_test = regression_calibration_plotly(cal_test, title="Калибровка test")
fig_cal_oot = regression_calibration_plotly(cal_oot, title="Калибровка oot")

In [ ]:
metrics_summary.columns

In [ ]:
metrics_summary[['Dataset', 'N_samples', 'N_feat', 'avg_target', 'avg_pred', 'MAE', 'RMSE', 'R2', 'MAE_95pct']]

In [ ]:
# Создаём отчёт 
report_html = report_generator.Report_HTML(
    title=TITLE,
    description=DESCRIPTION,
    footer=f"Дата создания отчёта: {datetime.date.today().strftime('%d-%m-%Y')}" 
)

# === ВКЛАДКА 1: СПЕЦИФИКАЦИЯ МОДЕЛИ ===
spec_tab = report_html.add_tab("Спецификация модели")

# Строка 1x2: Основная информация
row1 = spec_tab.add_row("1x2")

# Основные характеристики
model_specs = {
    "Название": TITLE,
    "Тип задачи": ML_TYPE,
    "Режим работы": MODE,
    "Регрессия": MODEL_TYPE,
    "Версия модели": SUBTASK,
    "Разработчик модели": AUTHOR
}
row1.cells[0].add_kv(
    model_specs,
    title="Основные характеристики",
    caption="Ключевая информация о модели"
)

# Данные и выборки
data_specs = {
    "Источник данных": OUT_TABLE,
    "Популяция": POPULATION_TEXT,
    "Период обучения": TRAIN_TEST_PERIOD,
    "Период OOT": OOT_PERIOD,
}
row1.cells[1].add_kv(
    data_specs,
    title="Данные и выборки", 
    caption="Информация об источниках данных и временных периодах"
)

# Строка 1x2: Основная метрики
row2 = spec_tab.add_row(['1fr', '2fr'])
model_specs = best_model.get_params()

row2.cells[0].add_kv(
    model_specs,
    title="Параметры модели", 
    caption="Подобранные гиперпараметры"
)

rows = []
for name, df in [('Train', train), ('Test', test), ('OOT', oot)]:
    y = pd.to_numeric(df[TARGET], errors='coerce')
    total = len(df)
    above_zero = (y > 0).sum()
    mean_target = y.mean()
    rows.append([
        name,
        total,
        int(above_zero),
        f'{mean_target:.4f}',  # или :.2f — как удобнее
    ])

cnt_df = pd.DataFrame(
    rows,
    columns=['Выборка','Количество контрактов, шт.','Таргет > 0, шт.','Среднее по таргету',],
)

row2.cells[1].add_dataframe(
    cnt_df,
    title="Кол-во контрактов и уровень таргета", 
    caption="Информация о выборке"
)

# === ВКЛАДКА 2: ВЫБОРКИ ===
sample_dfs_tab = report_html.add_tab("Данные для построения")

# Строка 1x2: График sample
row1 = sample_dfs_tab.add_row(['2fr', '1fr'])
row1.cells[0].add_plotly_figure(
    fig_sample_try,
    title="Динамика таргета во времени", 
    caption="Информация о выборке"
)
row1.cells[1].add_dataframe(
    info_sample_try[['month', 'count', 'mean']],
    title="Динамика таргета во времени", 
    caption="Информация о выборке"
)

row2 = sample_dfs_tab.add_row("1x1")
row2.cells[0].add_plotly_figure(
    fig_df ,
    title="Динамика таргета во времени", 
    # caption="Информация о Train/Test/OOT"
)
row3 = sample_dfs_tab.add_row("1x1")
row3.cells[0].add_dataframe(
    info_df,
    title="Динамика таргета во времени", 
    caption="Информация о Train/Test/OOT"
)

# === ВКЛАДКА 3: МЕТРИКИ МОДЕЛИ ===
model_metrics_tab = report_html.add_tab("Метрики модели")

row1 = model_metrics_tab.add_row("1x1")
row1.cells[0].add_dataframe(
    metrics_summary[['Dataset', 'N_samples', 'N_feat', 'avg_target', 'avg_pred', 'MAE', 'RMSE', 'R2', 'MAE_95pct']],
    title="Метрики качества", 
    caption="Показатели на Train/Test/OOT"
)
row2 = model_metrics_tab.add_row(['1fr', '2fr'])
row2.cells[0].add_plotly_figure(
    fig_month_mae,
    title="Динамика MAE во времени", 
)
row2.cells[1].add_dataframe(
    df_month[['month', 'cnt', 'avg_target', 'avg_pred', 'mae', 'r2']],
    title="Динамика Target во времени", 
)

row4 = model_metrics_tab.add_row("1x3")
for i, fig in enumerate([fig_lift_train, fig_lift_test, fig_lift_oot]):
    row4.cells[i].add_plotly_figure(
        fig,
        title="Lift")

row5 = model_metrics_tab.add_row("1x3")
row5.cells[0].add_dataframe(
        train_decile[['decile', 'client_all', 'mean_target',
       'mean_pred']],
        title="Таргет по децилям Train")
row5.cells[1].add_dataframe(
        test_decile[['decile', 'client_all', 'mean_target',
       'mean_pred']],
        title="Таргет по децилям Test")
row5.cells[2].add_dataframe(
        oot_decile[['decile', 'client_all', 'mean_target',
       'mean_pred']],
        title="Таргет по децилям OOT")


# === ВКЛАДКА 4: ФИЧИ И PSI ===
feat_tab = report_html.add_tab("Признаки и PSI")

row1 = feat_tab.add_row("1x1")
row1.cells[0].add_plotly_figure(
        psi_fig,
        title="PSI всех признаков модели по месяцам",)
row2 = feat_tab.add_row("1x1")
row2.cells[0].add_plotly_figure(
        psi_score_fig,
        title="PSI скоров модели по месяцам",)
row3 = feat_tab.add_row("1x1")
row3.cells[0].add_dataframe(
    pd.merge(imp_df, description_df, how='left', left_on='feature', right_on='f_name')[['importance', 
                                                                                      'feature', 
                                                                                      'f_desc']],
    title="FeatureImp и расшифровка признаков", 
    caption="Источник описания - SB_DIRKAIM.FEATURE_REGISTRY"
)

report_html.save(f'{REPORTS_FOLDER}/{MODEL_NAME}_report.html')

In [ ]:
df_month_f = df_month[['month', 'cnt', 'avg_target', 'avg_pred', 'mae', 'r2']]

In [ ]:
df_month_f

In [ ]:
df_month_f = df_month_f.style.format({
    'month': '{:%Y-%m-%d}',
    'avg_target': '{:.1f}',
    'avg_pred': '{:.1f}',
    'mae': '{:.1f}',
    'r2': '{:.3f}',
})

In [ ]:
df_month[['month', 'cnt', 'avg_target', 'avg_pred', 'mae', 'r2']]

In [ ]:
train_decile.columns

In [ ]:
train_decile[['decile', 'client_all', 'mean_target',
       'mean_pred']]

In [ ]:
train_decile['mean_target'] = round(train_decile['mean_target'], 2)
train_decile['mean_pred'] = round(train_decile['mean_pred'], 2)

test_decile['mean_target'] = round(test_decile['mean_target'], 2)
test_decile['mean_pred'] = round(test_decile['mean_pred'], 2)

oot_decile['mean_target'] = round(oot_decile['mean_target'], 2)
oot_decile['mean_pred'] = round(oot_decile['mean_pred'], 2)

## 10. Сохранение артефактов

In [ ]:
s3_secret = (await Secret.load("s3-retail")).get()

In [ ]:
ACCESS_KEY = s3_secret["access_key"]
SECRET_KEY = s3_secret["secret_key"]

In [ ]:
from minio import Minio

client = Minio(
    endpoint="10.167.8.142:9000",
    access_key=ACCESS_KEY,
    secret_key=SECRET_KEY,
    secure=False
)    

In [ ]:
def create_df(df, target, pred_col, y_pred, segment_name):
    result_df = df[PRIMARY_KEY+[target]].copy()
    result_df[pred_col] = y_pred
    result_df['segment'] = segment_name

    return result_df

all_psi = []
description_dfs = []
for target in [TARGET]:
    os.makedirs(f"{PATH_MODELS_DATA}/{target}/", exist_ok=True)
    best_features = results[target]['selected']
    train_df = create_df(train, target, f'{target}_pred', results[target]['model'].predict(train[best_features]), 'train')
    test_df = create_df(test, target, f'{target}_pred', results[target]['model'].predict(test[best_features]), 'test')
    oot_df = create_df(oot, target, f'{target}_pred', results[target]['model'].predict(oot[best_features]), 'oot')
    
    scored_df = pd.concat([train_df, test_df, oot_df], ignore_index=True)
    
    # сохраняем train, test, oot выборки после отбора фичей и проскоренную выборку
    train[PRIMARY_KEY + best_features + [target]].to_parquet(f"{PATH_MODELS_DATA}/{target}/sample_train.parquet")
    test[PRIMARY_KEY + best_features + [target]].to_parquet(f"{PATH_MODELS_DATA}/{target}/sample_test.parquet")
    oot[PRIMARY_KEY + best_features + [target]].to_parquet(f"{PATH_MODELS_DATA}/{target}/sample_oot.parquet")
    scored_df.to_parquet(f"{PATH_MODELS_DATA}/{target}/scored_df.parquet")

    psi_scores = preprocessing.calculate_feat_psi(scored_df[[f'{target}_pred', 'report_date']], [f'{target}_pred'], date_col='report_date')
    psi_scores = psi_scores.rename(columns={f'{target}_pred':'PSI'})
    psi_scores['target'] = target
    all_psi.append(psi_scores)

    psi_result[best_features].T.to_excel(f'{REPORTS_FOLDER}/psi_top20_{target}.xlsx')
    
    features_sql = ','.join(f"'{f}'" for f in best_features)

    engine_sb_dirkaim = oracle.create_engine_cdw(USERNAME_SB_DIRKAIM, PASSWORD_SB_DIRKAIM)
    with engine_sb_dirkaim.connect() as conn:
        description_df =  pd.read_sql(f"""select * from SB_DIRKAIM.FEATURE_REGISTRY where f_name in ({features_sql})""", conn)
    description_df = description_df[['f_name', 'f_desc', 'proc']]
    description_df['target'] = target
    description_dfs.append(description_df)

    client.fput_object('retail', 
                       f'models/eservices-{TASK_NUM}/{SUBTASK}/{target}/train.paruqet',
                       f'{PATH_MODELS_DATA}/{target}/sample_train.parquet')
    client.fput_object('retail', 
                       f'models/eservices-{TASK_NUM}/{SUBTASK}/{target}/test.paruqet',
                       f'{PATH_MODELS_DATA}/{target}/sample_test.parquet')
    client.fput_object('retail', 
                       f'models/eservices-{TASK_NUM}/{SUBTASK}/{target}/oot.paruqet', 
                       f'{PATH_MODELS_DATA}/{target}/sample_oot.parquet')
    client.fput_object('retail', 
                       f'models/eservices-{TASK_NUM}/{SUBTASK}/{target}/scored_df.paruqet', 
                       f'{PATH_MODELS_DATA}/{target}/scored_df.parquet')

In [ ]:
psi_pred = pd.concat(all_psi, ignore_index=True)
descriptions = pd.concat(description_dfs, ignore_index=True)

## 12. Публикация модели в Model_Registry

In [ ]:
# Креды для подключения S3
s3_kwargs = (await Secret.load("s3-retail")).get()
# Пароль для работы с SB_DIRKAIM (!)
# cdw_pass = (await Secret.load("pass-sb-dirkaim")).get() 


from model_repository import ModelRepository
from model_repository.dao import OracleDAO
from model_repository.storage import MinioClient

engine = oracle.create_engine_cdw(
    USERNAME_SB_DIRKAIM, PASSWORD_SB_DIRKAIM
)
dao = OracleDAO(engine)
storage = MinioClient(**s3_kwargs)

# Инициализируем класс ModelRepository
repo = ModelRepository(storage=storage, dao=dao)

In [ ]:
from model_repository import RegisterRequest

# допустим, у тебя есть sklearn/catboost/lightgbm модель в памяти: best_model
req = RegisterRequest(
    model_name=MODEL_NAME,
    model_type="lightgbm",  # "catboost" | "lightgbm" | "sklearn" | "other"
    artifact_format="pickle",
    uri=f"retail/models/eservices-{TASK_NUM}/{SUBTASK}/{MODEL_NAME}.pkl",
    created_by=AUTHOR,
    description="ОД по контракту на 6 месяц", # Описание целевой переменной
    scoring_flg=0,  # Флаг что модель на активном скоринге
    monitoring_flg=0,  # Флаг что модель на мониторинге
    is_online=0,  # Флаг онлайн модели
    model_obj=best_model,  # авто-сериализуем, авто-извлечём важности
)

In [ ]:
# Регистрируем модель
meta = repo.register(req)
print(meta)

In [ ]:
# Проверка что модель зарегистрирована
assert repo.exists(MODEL_NAME)